# Writeup Tables — Generated from `4_neuron_list` Files

This notebook computes the three tables included in the writeup (Sections 7.3, 7.4, 7.5)
directly from the 9 result files produced by `4_neuron_list.ipynb`.

In [ ]:
import pandas as pd
import os
from pathlib import Path

# ---------- Configuration ----------
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    DATA_DIR = Path("/content/drive/MyDrive/DATA/New-Atlas")
else:
    DATA_DIR = Path("/Users/piotrwilam/Data/New-Atlas")

EPSILONS = ["0.001", "0.1", "0.5"]

LANG = "P"
MODEL = "QW"
PREFIX = f"{LANG}_{MODEL}_"
CONSISTENCIES = ["0.2", "0.5", "0.8"]
N_LAYERS = 28

# Layers string matching 4_neuron_list output (all layers)

def _fname(eps: str, cons: str) -> str:
    return f"{PREFIX}4_neuron_list_eps{eps}_cons{cons}_all_layers_both.csv"

# ---------- Object classification ----------
MODULAR_ASTS = {
    "ast__Import", "ast__Break", "ast__Pass",
    "ast__ImportFrom", "ast__Continue", "ast__Assert",
}

def classify(obj: str) -> str:
    """Classify an object into one of four groups."""
    if obj.startswith("builtin__"):
        return "Builtin"
    if obj in MODULAR_ASTS:
        return "Modular AST"
    return "Non-modular AST"

# ---------- Load all 9 files ----------
frames = {}
for eps in EPSILONS:
    for cons in CONSISTENCIES:
        path = DATA_DIR / _fname(eps, cons)
        if not path.exists():
            print(f"WARNING: missing {path.name}")
            continue
        df = pd.read_csv(path)
        df["circuit_size"] = df["n_concept_only"] + df["n_both"]
        df["group"] = df["object"].map(classify)
        frames[(eps, cons)] = df

print(f"Loaded {len(frames)} of 9 files.")
print(f"Data dir: {DATA_DIR}")

## Table 1 — Section 7.3: Concept fraction across the 3×3 parameter grid

Aggregate concept fraction = Σ concept-only / Σ circuit_size, computed separately
for AST objects (all testable) and builtin objects (all testable).

In [ ]:
rows = []
for eps in EPSILONS:
    for cons in CONSISTENCIES:
        df = frames.get((eps, cons))
        if df is None:
            rows.append({"ε": eps, "Consistency": cons,
                         "AST concept fraction": None,
                         "Builtin concept fraction": None})
            continue
        ast_df = df[df["object"].str.startswith("ast__")]
        blt_df = df[df["object"].str.startswith("builtin__")]

        ast_cf = ast_df["n_concept_only"].sum() / ast_df["circuit_size"].sum() if ast_df["circuit_size"].sum() > 0 else 0
        blt_cf = blt_df["n_concept_only"].sum() / blt_df["circuit_size"].sum() if blt_df["circuit_size"].sum() > 0 else 0

        rows.append({
            "ε": eps,
            "Consistency": cons,
            "AST concept fraction": f"{ast_cf:.1%}",
            "Builtin concept fraction": f"{blt_cf:.1%}",
        })

table1 = pd.DataFrame(rows)
print("Table 1 — Section 7.3: Concept fraction across parameter grid")
print()
print(table1.to_string(index=False))

## Table 2 — Section 7.4: Loudest neurons (ε=0.5, consistency 80%, mid layer)

Groups: Modular ASTs, Non-modular ASTs, Builtins.
Uses the mid-network layer (layer 14 of 28 ≈ layer 5 of 8 in CSP-Atlas).

In [ ]:
MID_LAYER = 14  # Approximate mid-network layer for 28-layer model

df_loud = frames.get(("0.5", "0.8"))
if df_loud is None:
    print("ERROR: ε=0.5 cons=0.8 file not loaded.")
else:
    l_mid = df_loud[df_loud["layer"] == MID_LAYER].copy()

    # Count objects per group
    n_mod = l_mid[l_mid["group"] == "Modular AST"]["object"].nunique()
    n_nonmod = l_mid[l_mid["group"] == "Non-modular AST"]["object"].nunique()
    n_blt = l_mid[l_mid["group"] == "Builtin"]["object"].nunique()

    rows2 = []
    for group_name, group_objects in [
        (f"Modular ASTs ({n_mod})", l_mid[l_mid["group"] == "Modular AST"]),
        (f"Non-modular ASTs ({n_nonmod})", l_mid[l_mid["group"] == "Non-modular AST"]),
        (f"Builtins ({n_blt})", l_mid[l_mid["group"] == "Builtin"]),
    ]:
        c = group_objects["n_concept_only"].sum()
        s = group_objects["n_both"].sum()
        sz = c + s
        cf = c / sz if sz > 0 else 0
        rows2.append({
            "Group": group_name,
            "Concept-only": c,
            "Shared": s,
            "Circuit size": sz,
            "Concept fraction": f"{cf:.1%}",
        })

    table2 = pd.DataFrame(rows2)
    print(f"Table 2 — Section 7.4: Loudest neurons (ε=0.5, cons=0.8, L{MID_LAYER})")
    print()
    print(table2.to_string(index=False))

## Table 3 — Section 7.5: Layer profile of concept signal (ε=0.001, consistency 80%)

Concept fraction per layer for Modular ASTs, Non-modular ASTs, and Builtins.

In [ ]:
df_base = frames.get(("0.001", "0.8"))
if df_base is None:
    print("ERROR: ε=0.001 cons=0.8 file not loaded.")
else:
    available_layers = sorted(df_base["layer"].unique())
    
    rows3 = []
    for group_label in ["Modular AST", "Non-modular AST", "Builtin"]:
        row = {"Group": group_label}
        for layer in available_layers:
            sub = df_base[(df_base["layer"] == layer) & (df_base["group"] == group_label)]
            c = sub["n_concept_only"].sum()
            sz = sub["circuit_size"].sum()
            cf = c / sz if sz > 0 else 0
            row[f"L{layer}"] = f"{cf:.1%}"
        rows3.append(row)

    table3 = pd.DataFrame(rows3)
    print("Table 3 — Section 7.5: Layer profile (ε=0.001, cons=0.8)")
    print()
    print(table3.to_string(index=False))

## Markdown output

Print all three tables in markdown format, ready to paste into the writeup.

In [ ]:
def to_md(df: pd.DataFrame) -> str:
    """Convert a DataFrame to a markdown table string."""
    cols = df.columns.tolist()
    lines = ["| " + " | ".join(cols) + " |"]
    lines.append("|" + "|".join(["---"] * len(cols)) + "|")
    for _, row in df.iterrows():
        lines.append("| " + " | ".join(str(row[c]) for c in cols) + " |")
    return "\n".join(lines)

print("### Table 1 — Section 7.3")
print()
print(to_md(table1))
print()
print()
print("### Table 2 — Section 7.4")
print()
if 'table2' in dir():
    print(to_md(table2))
else:
    print("(not available)")
print()
print()
print("### Table 3 — Section 7.5")
print()
if 'table3' in dir():
    print(to_md(table3))
else:
    print("(not available)")